## Practical 1

# Import required libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Load Dataset

In [15]:
df = pd.read_csv('../data/heart_disease_uci.csv')
print("Original Dataset Shape:", df.shape)
print("\nColumns found in your dataset:", df.columns.tolist())

# Convert any '?' to actual NaNs.
df.replace('?', np.nan, inplace=True)
df.head()

Original Dataset Shape: (920, 16)

Columns found in your dataset: ['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


# 2. Handle missing values (Mean for Age, Mode for Embarked)

In [16]:
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = SimpleImputer(strategy='most_frequent').fit_transform(df[[col]]).ravel()
        else:
            df[col] = SimpleImputer(strategy='mean').fit_transform(df[[col]]).ravel()

# 3. Handle Categorical Data (One-Hot Encoding)

In [20]:

potential_cat_cols = ['cp', 'thal', 'slope', 'restecg', 'chest_pain_type', 'resting_ecg', 'st_slope']
existing_cat_cols = [col for col in potential_cat_cols if col in df.columns]

if existing_cat_cols:
    df = pd.get_dummies(df, columns=existing_cat_cols, drop_first=True)

# 4. Feature Scaling (Standardization)

In [19]:
scaler = StandardScaler()

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for target_name in ['target', 'num', 'condition']:
    if target_name in numeric_cols:
        numeric_cols.remove(target_name)

df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

df.to_csv('../data/heart_clean.csv', index=False)

print("\nPreprocessed Dataset Preview:\n", df.head())


Preprocessed Dataset Preview:
          id       age     sex    dataset  trestbps      chol    fbs    thalch  \
0 -1.730169  1.007386    Male  Cleveland  0.698041  0.311021   True  0.495698   
1 -1.726404  1.432034    Male  Cleveland  1.511761  0.797713  False -1.175955   
2 -1.722639  1.432034    Male  Cleveland -0.658158  0.274289  False -0.340128   
3 -1.718873 -1.752828    Male  Cleveland -0.115679  0.467130  False  1.968345   
4 -1.715108 -1.328180  Female  Cleveland -0.115679  0.044717  False  1.371326   

   exang   oldpeak  ...  num  cp_atypical angina  cp_non-anginal  \
0  False  1.349421  ...    0               False           False   
1   True  0.589832  ...    2               False           False   
2   True  1.634267  ...    1               False           False   
3  False  2.488805  ...    0               False            True   
4  False  0.494884  ...    0                True           False   

   cp_typical angina  thal_normal  thal_reversable defect  slope_flat  \